# PaliGemma 2 (3B mix 448) — Interactive inference sandbox

Loads `google/paligemma2-3b-mix-448` exactly as the production pipeline does
and exposes a single editable cell with `PROMPT` + `IMAGE_PATH` so you can
iterate on prompts (e.g. PDDL predicate extraction) without reloading the
model.

## How to use

1. **Runtime → GPU** (T4 free is enough for 3B / bf16).
2. Run **§1** (install) and **§2** (load model). The model load takes ~30 s
   the first time and stays in VRAM until you restart the runtime.
3. Upload your test image to `/content/` (drag-and-drop in the file panel).
4. Edit `IMAGE_PATH` and `PROMPT` in **§3** and run that cell. Re-run §3 as
   many times as you like — §1 and §2 don't need to be re-executed.

## What this notebook does NOT do

It does not score the answer, log to CSV, or run multiple prompts at once.
It's a manual exploration tool. For systematic evaluation use
`experiment1_paligemma.ipynb` / `experiment2_paligemma.ipynb`.


## 1. Install dependencies

If this is a fresh runtime, run this cell once and then `Runtime → Restart
runtime` (Pillow gets force-reinstalled, the kernel needs to reload it). On
restart you can skip §1 and start at §2.


In [ ]:
!pip install -qU --force-reinstall Pillow
!pip install -qU transformers accelerate
print()
print('=' * 60)
print('FIRST RUN ON FRESH RUNTIME: Runtime -> Restart runtime now,')
print('then run cells from sec 2 onwards (do NOT re-run sec 1).')
print('=' * 60)


## 2. Load model and processor

Loads PaliGemma 2 (3B mix 448) once. Uses bfloat16 on CUDA (~6 GB VRAM) to
match the production pipeline. After this cell completes, `model` and
`processor` stay in memory until the runtime restarts — so §3 can be edited
and re-run cheaply.

Gated model: set `HF_TOKEN` below with a token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) that
has accepted the [model terms](https://huggingface.co/google/paligemma2-3b-mix-448).


In [ ]:
import torch
from PIL import Image
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor

HF_TOKEN = ''  # paste your HuggingFace read token here
MODEL_ID = 'google/paligemma2-3b-mix-448'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

print(f'Loading {MODEL_ID} on {DEVICE} ({DTYPE})...')
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=DTYPE,
    device_map='auto' if DEVICE == 'cuda' else None,
).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded -- {n_params / 1e9:.2f}B params')
if DEVICE == 'cuda':
    print(f'VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')


## 3. Inference (edit IMAGE_PATH and PROMPT_ID, re-run this cell)

The four prompts mirror exactly what the on-device VLM (`vlm_smoke_test`) uses
in production, so the output here tells you what PaliGemma would produce on
the same image:

| `PROMPT_ID` | Use when image shows... | Expected output format |
|---|---|---|
| `door_pddl` | A classroom door plate (e.g. `1.7`, `IA`) | `(at user1 classroom_1_7)` |
| `sign_pddl` | A directory sign in building G | `(at user1 directory_sign_g_0)` |
| `stairs_text` | A staircase | `stairs up` or `stairs down` |
| `elevator_json_cabin` | An elevator (doors visible) | `{"type": "elevator", "detail": "open"}` |

Decoding stays greedy (`do_sample=False, num_beams=1`) so the response is
deterministic for the chosen (image, prompt) pair. Re-run §3 as many times as
you want — `model` and `processor` stay in VRAM.


In [ ]:
# === EDIT THESE TWO ===
IMAGE_PATH = '/content/your_image.jpg'  # drag-and-drop your image into /content/

# Pick one of: 'door_pddl', 'sign_pddl', 'stairs_text', 'elevator_json_cabin'
PROMPT_ID = 'door_pddl'

MAX_NEW_TOKENS = 60
# ======================

# Prompts copied verbatim from vlm_smoke_test/lib/prompts.dart so this notebook
# evaluates exactly what the on-device VLM is asked in production.
PROMPTS = {
    'door_pddl': (
        'What room number is on the plate in this image?
'
        '
'
        'Read the number carefully. It contains a DOT in the middle.
'
        'The DOT separates two digits. Keep both digits and put _ where the DOT is.
'
        '
'
        'Plate says 3.7 -> output: (at user1 classroom_3_7)
'
        'Plate says 0.2 -> output: (at user1 classroom_0_2)
'
        'Plate says IA  -> output: (at user1 lab_ia)
'
        '
'
        'Output ONLY the (at user1 ...) line.'
    ),
    'sign_pddl': (
        'Look at this image. It shows a large white sign on a wall inside building G.
'
        '
'
        'In the top-right area of the sign there are two lines of text, one above the other:
'
        '- Top line: the building letter (G)
'
        '- Bottom line: either a floor number (0, 1, or 2) OR the word "Agora"
'
        '
'
        'Read the bottom line and map it to the answer:
'
        '- "0" -> (at user1 directory_sign_g_0)
'
        '- "1" -> (at user1 directory_sign_g_1)
'
        '- "2" -> (at user1 directory_sign_g_2)
'
        '- "Agora" -> (at user1 directory_sign_g_agora)
'
        '
'
        'Output ONLY the (at user1 ...) line. Nothing else.'
    ),
    'stairs_text': (
        'Look at this image. It shows a staircase.
'
        '
'
        'Question: from the point of view of the camera, do the stairs go UP or DOWN?
'
        '
'
        'How to decide:
'
        '- If the next step is higher than the camera and the stairs rise away from you -> up
'
        '- If the next step is lower than the camera and the stairs drop away from you -> down
'
        '
'
        'Answer with exactly one of these two phrases:
'
        'stairs up
'
        'stairs down
'
        '
'
        'Output ONLY the phrase. No punctuation, no extra words.'
    ),
    'elevator_json_cabin': (
        'Look at this image. Focus on the elevator doors.
'
        '
'
        'Can you see the inside of the elevator cabin?
'
        '- YES, I can see inside the cabin -> the doors are open
'
        '- NO, I cannot see inside, the doors are a solid surface -> the doors are closed
'
        '
'
        'Valid answers:
'
        '{"type": "elevator", "detail": "open"}
'
        '{"type": "elevator", "detail": "closed"}
'
        '
'
        'Output ONLY the JSON. Nothing else.'
    ),
}

if PROMPT_ID not in PROMPTS:
    raise ValueError(f'Unknown PROMPT_ID {PROMPT_ID!r}. Pick one of: {list(PROMPTS)}')
PROMPT = PROMPTS[PROMPT_ID]

image = Image.open(IMAGE_PATH).convert('RGB')

# Display the image inline so you can confirm it's the right one.
try:
    from IPython.display import display
    display(image.resize((min(448, image.width), min(448, image.height))))
except Exception:
    pass

# Build inputs exactly as the production pipeline does (raw <image>{prompt}
# format, no chat template - PaliGemma is a VQA model, not chat-tuned).
full_prompt = f'<image>{PROMPT}'
inputs = processor(
    text=full_prompt,
    images=image,
    padding='longest',
    do_convert_rgb=True,
    return_tensors='pt',
).to(model.device)
inputs = inputs.to(dtype=model.dtype)

gen_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    num_beams=1,
    pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
    eos_token_id=processor.tokenizer.eos_token_id,
)

import time
if torch.cuda.is_available():
    torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad():
    output = model.generate(**inputs, **gen_kwargs)
if torch.cuda.is_available():
    torch.cuda.synchronize()
latency_ms = (time.time() - t0) * 1000

input_len = inputs['input_ids'].shape[-1]
response = processor.decode(output[0][input_len:], skip_special_tokens=True).strip()

print('=' * 60)
print(f'PROMPT_ID: {PROMPT_ID}')
print(f'IMAGE:     {IMAGE_PATH}')
print('-' * 60)
print('PROMPT:')
print(PROMPT)
print('-' * 60)
print('RESPONSE:')
print(response)
print('-' * 60)
print(f'Latency: {latency_ms:.1f} ms  |  Output tokens: {output.shape[1] - input_len}')
print('=' * 60)
